<a href="https://colab.research.google.com/github/Misbah361/60-days-AI-challenge/blob/main/Day-11-Retrieval-Engine/Day11_Retrieval_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ---------- Day 10 classes ----------
class PreprocessingModule:
    def __init__(self, use_lemma=True):
        self.use_lemma = use_lemma
        self.stop_words = set(stopwords.words('english'))
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()

    def tokenize(self, text):
        return word_tokenize(text)

    def remove_stopwords(self, tokens):
        lowered = [t.lower() for t in tokens]
        return [t for t in lowered if t not in self.stop_words]

    def strip_punct(self, tokens):
        cleaned = [re.sub(r'[^a-z0-9\s]', '', t) for t in tokens]
        return [t for t in cleaned if t.strip()]

    def normalise(self, tokens):
        if self.use_lemma:
            return [self.lemmatizer.lemmatize(t) for t in tokens]
        return [self.stemmer.stem(t) for t in tokens]

    def transform(self, text):
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords(tokens)
        tokens = self.strip_punct(tokens)
        tokens = self.normalise(tokens)
        return ' '.join(tokens)


class VectorizerModule:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.doc_matrix = None

    def fit(self, corpus):
        self.doc_matrix = self.vectorizer.fit_transform(corpus)
        return self

    def transform(self, query):
        q_vec = self.vectorizer.transform([query])
        return cosine_similarity(q_vec, self.doc_matrix).flatten()


# ---------- Day 11: knowledge base (22 documents) ----------
corpus = [
    "The Sun is a star at the center of our solar system and provides light and heat to Earth.",
    "Mars is called the red planet because iron oxide on its surface gives it a reddish color.",
    "Jupiter is the largest planet in the solar system and has a giant storm called the Great Red Spot.",
    "Saturn is famous for its bright rings made of ice and rock particles.",
    "The Moon orbits Earth and causes ocean tides through its gravitational pull.",
    "A black hole is a region of space where gravity is so strong that nothing, not even light, can escape.",
    "A galaxy is a huge collection of stars, gas, and dust held together by gravity; ours is the Milky Way.",
    "NASA astronauts traveled to the Moon during the Apollo missions between 1969 and 1972.",
    "The International Space Station orbits Earth and hosts astronauts who conduct scientific experiments.",
    "A light year is the distance light travels in one year, about nine trillion kilometers.",
    "Comets are icy bodies that develop glowing tails when they approach the Sun.",
    "Asteroids are rocky objects, mostly found in the belt between Mars and Jupiter.",
    "The Big Bang theory says the universe began from an extremely hot and dense state about 13.8 billion years ago.",
    "Telescopes such as Hubble and James Webb let scientists observe distant galaxies and stars.",
    "A solar eclipse happens when the Moon passes between the Sun and Earth and blocks sunlight.",
    "Rockets burn fuel to produce thrust and escape Earth's gravity to reach orbit.",
    "Venus is the hottest planet because its thick atmosphere traps heat in a runaway greenhouse effect.",
    "Neptune is a cold, blue ice giant with the fastest winds in the solar system.",
    "Exoplanets are planets that orbit stars outside our solar system and are found using transit methods.",
    "Stars are born in nebulae, shine by nuclear fusion, and may end as white dwarfs, neutron stars or black holes.",
    "ISRO launched the Chandrayaan-3 mission and landed a spacecraft near the Moon's south pole in 2023.",
    "Satellites orbit Earth to provide communication, weather forecasting and navigation services.",
]

# ---------- Preprocess + vectorise (fit once) ----------
pre = PreprocessingModule()
vec = VectorizerModule()
clean_corpus = [pre.transform(d) for d in corpus]
vec.fit(clean_corpus)
corpus_matrix = vec.doc_matrix
print("TF-IDF matrix shape:", corpus_matrix.shape)

# ---------- retrieve() with relevance threshold ----------
THRESHOLD = 0.1

def retrieve(query, corpus_matrix, top_k=3, threshold=THRESHOLD):
    if not isinstance(query, str) or not query.strip():
        return "No relevant document found (empty query)."
    clean_q = pre.transform(query)
    if not clean_q:
        return "No relevant document found (no meaningful words)."
    q_vec = vec.vectorizer.transform([clean_q])
    scores = cosine_similarity(q_vec, corpus_matrix).flatten()
    if scores.max() < threshold:
        return "No relevant document found."
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i]), corpus[i]) for i in top_idx]

# ---------- 10 test queries ----------
queries = [
    ("Which planet has rings?", "clear"),
    ("What is a black hole?", "clear"),
    ("How do rockets reach orbit?", "clear"),
    ("Tell me about the red planet", "clear"),
    ("Chandrayaan Moon landing", "clear"),
    ("star", "ambiguous"),
    ("What is the largest thing?", "ambiguous"),
    ("How do I bake chocolate cake?", "out-of-domain"),
    ("Who won the cricket world cup?", "out-of-domain"),
    ("Which frozen wanderer leaves a luminous trail?", "synonym"),
]

for q, kind in queries:
    print("=" * 70)
    print(f"[{kind.upper()}] Query: {q}")
    res = retrieve(q, corpus_matrix, top_k=3)
    if isinstance(res, str):
        print("  ->", res)
    else:
        for rank, (i, s, t) in enumerate(res, 1):
            print(f"  {rank}. (doc {i}, score={s:.3f}) {t[:75]}...")

TF-IDF matrix shape: (22, 167)
[CLEAR] Query: Which planet has rings?
  1. (doc 3, score=0.289) Saturn is famous for its bright rings made of ice and rock particles....
  2. (doc 18, score=0.151) Exoplanets are planets that orbit stars outside our solar system and are fo...
  3. (doc 2, score=0.148) Jupiter is the largest planet in the solar system and has a giant storm cal...
[CLEAR] Query: What is a black hole?
  1. (doc 5, score=0.431) A black hole is a region of space where gravity is so strong that nothing, ...
  2. (doc 19, score=0.341) Stars are born in nebulae, shine by nuclear fusion, and may end as white dw...
  3. (doc 20, score=0.000) ISRO launched the Chandrayaan-3 mission and landed a spacecraft near the Mo...
[CLEAR] Query: How do rockets reach orbit?
  1. (doc 15, score=0.545) Rockets burn fuel to produce thrust and escape Earth's gravity to reach orb...
  2. (doc 4, score=0.117) The Moon orbits Earth and causes ocean tides through its gravitational pull...
  3. (doc 21

## Day 11: Failure Analysis

**Query 4: "Tell me about the red planet"** returned Jupiter (0.390) above Mars (0.385).
Diagnosis: Jupiter's document also contains "red" and "planet", so it scores almost the same as Mars, and TF-IDF cannot tell that "red planet" means Mars.

**Query 5: "Chandrayaan Moon landing"** returned generic Moon documents instead of the ISRO Chandrayaan-3 document.
Diagnosis: The Day 10 preprocessing turns "Chandrayaan-3" into the single token "chandrayaan3", which never matches the query word "chandrayaan".

**Query 7: "What is the largest thing?"** returned only Jupiter with a real score.
Diagnosis: "thing" appears in no document, so "largest" alone decides the result, and other large objects such as the galaxy and the universe are never considered.

**Query 10: "Which frozen wanderer leaves a luminous trail?"** returned "No relevant document found", although the comet document exists.
Diagnosis: The query shares no words with the comet document, which says "icy", "glowing" and "tails", so every score is below the 0.1 threshold.

## Relevance Threshold
Queries 8 and 9 (baking a cake, the cricket world cup) are out of domain. Their highest score was below 0.1, so the system correctly answered "No relevant document found" instead of returning a low-quality result.

## Vocabulary Mismatch and the Need for Embeddings
TF-IDF only matches exact tokens (after lemmatisation). The query "frozen wanderer leaves a luminous trail" means the same as "comets are icy bodies with glowing tails", but it shares no words with that document, so TF-IDF scores it 0 and fails. It has no idea that "frozen" and "icy", or "luminous" and "glowing", are related. The threshold turned this failure into an honest "not found" instead of a wrong answer, but the information was in the knowledge base all along. Embeddings place words and sentences with similar meanings close together in vector space, so they can retrieve the comet document without any shared words. This is why modern retrieval systems use embeddings, or combine them with TF-IDF.